In [ ]:
import os

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
import csv
import json
import os
import asyncio
import aiohttp
import time

API_KEY = os.environ["GOOGLE_API_KEY"]
MODEL = "gemini-2.5-flash"

INPUT_FILE = "sample_inventory_500.txt"
OUTPUT_FILE = "car_descriptions_parallel.json"

# -----------------------------
# Utility functions
# -----------------------------
def load_cars(file_path):
    cars = []
    with open(file_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="|")
        for row in reader:
            clean = {k.strip(): v.strip() for k, v in row.items() if k and v}
            if clean:
                cars.append(clean)
    return cars

def load_results():
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

async def generate_description(session, car, semaphore):
    """Generate a car description using Gemini API asynchronously."""
    vin = car.get("VIN")

    prompt = f"""
    Write a short, SEO-friendly description (3–4 sentences) for:
    {car.get('Year')} {car.get('Make')} {car.get('Model')} {car.get('Trim')},
    {car.get('Mileage')} km, {car.get('FuelType')} {car.get('Transmission')},
    {car.get('BodyType')} in {car.get('Color')} color, {car.get('Condition')} condition.
    Emphasize key selling points naturally.
    """

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={API_KEY}"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}

    async with semaphore:
        for attempt in range(5):
            try:
                async with session.post(url, json=payload, timeout=90) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        desc = data["candidates"][0]["content"]["parts"][0]["text"].strip()
                        print(f"✅ {car['Make']} {car['Model']}")
                        return {"VIN": vin, "Make": car["Make"], "Model": car["Model"], "Description": desc}
                    elif resp.status in (429, 503):
                        wait = 10 * (attempt + 1)
                        print(f"⏳ {vin}: {resp.status} – waiting {wait}s...")
                        await asyncio.sleep(wait)
                    else:
                        text = await resp.text()
                        print(f"⚠️ Error {resp.status}: {text}")
                        break
            except Exception as e:
                print(f"⚠️ Exception for {vin}: {e}")
                await asyncio.sleep(10)
        return None

# -----------------------------
# Main parallel runner
# -----------------------------
async def main():
    cars = load_cars(INPUT_FILE)
    results = load_results()
    completed_vins = {r["VIN"] for r in results}

    print(f"Loaded {len(cars)} cars, resuming from {len(completed_vins)} done")

    tasks = []
    semaphore = asyncio.Semaphore(3)  # limit concurrent API calls (safe for free-tier)

    async with aiohttp.ClientSession() as session:
        for car in cars:
            if car["VIN"] not in completed_vins:
                tasks.append(generate_description(session, car, semaphore))

        batch_size = 10
        for i in range(0, len(tasks), batch_size):
            batch = tasks[i:i+batch_size]
            print(f"\n🚀 Processing batch {i//batch_size + 1}/{len(tasks)//batch_size + 1}...")
            results_batch = await asyncio.gather(*batch)
            results_batch = [r for r in results_batch if r]
            results.extend(results_batch)

            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

            print(f"💾 Saved batch {i//batch_size + 1}, total = {len(results)}")
            await asyncio.sleep(15)  # cooldown between batches

    print(f"\n✅ Done! All results saved to {OUTPUT_FILE}")

# Run


if __name__ == "__main__":
    try:
        # If already in an event loop (like Jupyter)
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        # In Jupyter — use create_task
        import nest_asyncio
        nest_asyncio.apply()
        task = loop.create_task(main())
    else:
        asyncio.run(main())

    
